# 4 · HistGradientBoosting — Predicting Flight Delays

`HistGradientBoostingClassifier` is scikit-learn's own gradient-boosted tree. It uses the **same histogram-based split-finding** trick as LightGBM / XGBoost's `tree_method='hist'`, so it's fast and memory-efficient, and it **natively supports categorical features** without one-hot encoding.

It's a great choice when you want boosting performance without a third-party dependency.

## Setup

**Required packages**
```
pip install pandas numpy matplotlib seaborn scikit-learn xgboost
```
The dataset is `bwi_model_ready.csv` — already cleaned by `preprocess.py`.
It covers 14 days at Baltimore/Washington International (BWI), joined with hourly weather.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 5)

CATEGORICAL = ["carrier", "origin_airport", "destination_airport", "direction"]

df = pd.read_csv("bwi_model_ready.csv")
for col in CATEGORICAL:
    df[col] = df[col].astype("category")

print("shape:", df.shape)
df.head()


### Split (categoricals stay as `category` dtype)
We pass `categorical_features="from_dtype"` to the model and it auto-detects the category columns directly from the DataFrame.

In [ ]:
# Target: binary "is the flight delayed >= 15 min?"
# We drop rows where the label is missing (cancelled flights with no arrival time).
data = df.dropna(subset=["is_delayed"]).copy()
data["is_delayed"] = data["is_delayed"].astype(int)

TARGET = "is_delayed"
DROP_FROM_X = ["is_cancelled", "arrival_delay_minutes", "is_delayed"]
feature_cols = [c for c in data.columns if c not in DROP_FROM_X]

# Time-based split: BTS rows come pre-sorted by date.
# Using a chronological holdout is the right call for this kind of data —
# a random split would leak "future" information into the training set.
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]
X_train, y_train = train[feature_cols], train[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]

print(f"train rows: {len(X_train):,}   positive rate: {y_train.mean():.1%}")
print(f"test  rows: {len(X_test):,}   positive rate: {y_test.mean():.1%}")


### Fit the model
Key hyperparameters:
- `max_iter=400` with early stopping
- `learning_rate=0.05`
- `max_leaf_nodes=31` (leaf-wise growth; LightGBM-style)
- `class_weight="balanced"` for the minority class

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    categorical_features='from_dtype',
    class_weight='balanced',
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
    random_state=42,
)
model.fit(X_train, y_train)
print(f"iterations used: {model.n_iter_}")

### Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve,
)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

metrics = {
    "accuracy":  accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall":    recall_score(y_test, pred, zero_division=0),
    "f1":        f1_score(y_test, pred, zero_division=0),
    "roc_auc":   roc_auc_score(y_test, proba),
    "pr_auc":    average_precision_score(y_test, proba),
}
pd.Series(metrics).round(4).to_frame("value")


In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["on-time", "delayed"])
ax.set_yticklabels(["on-time", "delayed"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix")
plt.tight_layout(); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
prec, rec, _ = precision_recall_curve(y_test, proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate")
axes[0].legend()

axes[1].plot(rec, prec, label=f"AP = {metrics['pr_auc']:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray",
                label=f"baseline = {y_test.mean():.2f}")
axes[1].set(title="Precision–Recall curve", xlabel="Recall", ylabel="Precision")
axes[1].legend()
plt.tight_layout(); plt.show()


### Feature importance via permutation
`HistGradientBoosting` doesn't expose a `feature_importances_` attribute — the algorithm works over binned histograms, not per-feature gains. **Permutation importance** is the standard alternative: shuffle one column at a time and measure how much the test metric drops. A bigger drop means the model relied on that feature more.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model, X_test, y_test, scoring='average_precision',
    n_repeats=5, random_state=42, n_jobs=-1,
)
imp = (pd.Series(perm.importances_mean, index=feature_cols)
         .sort_values(ascending=False)
         .head(20))
ax = imp[::-1].plot.barh(color='darkorange')
ax.set(title='Top 20 permutation importances (PR-AUC drop)',
       xlabel='mean PR-AUC decrease')
plt.tight_layout(); plt.show()

### Training curve
`train_score_` / `validation_score_` are recorded when `early_stopping=True`.

In [ ]:
plt.plot(model.validation_score_, label='val score')
plt.axvline(model.n_iter_ - 1, color='red', ls='--',
            label=f'stopped at iter {model.n_iter_}')
plt.xlabel('boosting iteration'); plt.ylabel('loss (negative → better)')
plt.title('HGB early-stopping curve'); plt.legend()
plt.tight_layout(); plt.show()

### Bonus: regressing the delay magnitude

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

reg_data = df.dropna(subset=['arrival_delay_minutes']).copy()
sp = int(len(reg_data) * 0.8)
X_reg_tr, X_reg_te = reg_data.iloc[:sp][feature_cols], reg_data.iloc[sp:][feature_cols]
y_reg_tr, y_reg_te = reg_data.iloc[:sp]['arrival_delay_minutes'], reg_data.iloc[sp:]['arrival_delay_minutes']

reg = HistGradientBoostingRegressor(
    max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
    categorical_features='from_dtype', early_stopping=True,
    random_state=42,
)
reg.fit(X_reg_tr, y_reg_tr)
pred = reg.predict(X_reg_te)
print(f"MAE: {mean_absolute_error(y_reg_te, pred):.2f} min")
print(f"R²:  {r2_score(y_reg_te, pred):.3f}")

### Writeup prompts
- Put the PR-AUC of all three models (RF, XGBoost, HGB) in a single table. Which wins?
- Where do the models disagree on feature importance? Propose one explanation.
- Pick one misclassified test flight and describe, in plain English, why the model might have gotten it wrong.